# V14 第一性原理方案：可复现证据分析

**用途**：把社区机制、oracle、已暴露开发集、shadow QA、oracle-gap、package QA，以及文件存在后才会加入的 fresh screen / fresh confirm / Kaggle submission receipt，放进同一条可审计证据链。

**硬边界**：本 notebook 不读取 test outcome，不启动任何新对局，不执行 screen/confirm，不调用 Kaggle API，也不提交。所有指标都从已有报告的逐局 margin 重新计算或从 QA receipt 核验。

## TL;DR

结论由下方代码动态生成。只要 fresh validation audit 或 submission JSON 按约定落盘，重新从头执行即可升级证据层级；缺失时不会把 dev/oracle 伪装成 fresh confirm 或 online。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.venv').is_dir() and (candidate / 'kaggle_Kaggriculture').is_dir():
            return candidate
    raise RuntimeError('请从 DS_completation 仓库内执行 notebook')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
REPORTING_DIR = REPO_ROOT / 'kaggle_Kaggriculture/model/v14_first_principles_search/reporting'
if str(REPORTING_DIR) not in sys.path:
    sys.path.insert(0, str(REPORTING_DIR))

from v14_evidence import collect_evidence, compact_status

if Path(sys.prefix).resolve() != (REPO_ROOT / '.venv').resolve():
    raise RuntimeError(f'必须使用项目 .venv；当前解释器：{sys.executable}')

snapshot = collect_evidence()
print(f'解释器: {sys.executable}')
print(f'快照: {snapshot["generated_at"]}')
print(compact_status(snapshot))
print('边界:', json.dumps(snapshot['scope'], ensure_ascii=False, indent=2))

解释器: /Users/a1-6/Desktop/PycharmProjects/DS_completation/.venv/bin/python
快照: 2026-08-23T18:40:19.816387+00:00
online evidence present; inspect Kaggle status separately
Q2b exposed vs A2: 52/10/10 (pure 72.22%, score 79.17%); vs r002: 55/0/17 (pure 76.39%, score 76.39%).
边界: {
  "dates": [
    "2026-08-18",
    "2026-08-19",
    "2026-08-20"
  ],
  "allowed_splits": [
    "train",
    "validation"
  ],
  "test_outcomes_read": false,
  "new_games_started": false,
  "kaggle_api_called": false,
  "metric_definitions": {
    "pure_win_rate": "wins / all games; ties remain in denominator and are not wins",
    "competition_score_rate": "(wins + 0.5 * ties) / all games; report-only",
    "mean_margin": "candidate terminal reward minus anchor terminal reward"
  }
}


## Context & Methods

证据按可推翻性分层：`oracle_exposed` 只说明信息上限；`dev_exposed` 只用于开发和机制诊断；`shadow_qa` / `package_qa` 只证明实现或封装一致性；`fresh_screen` 用于一次性选择；`fresh_confirm` 才能支持本地提交门；`online` 仅来自本地保存的 Kaggle receipt。层级之间不可互相替代。

核心指标严格分开：

- **纯胜率** = W / (W + T + L)，平局留在分母中且不算胜；
- **competition score** = (W + 0.5T) / (W + T + L)，仅作比赛计分参考；
- W/T/L 与两个比率均由逐局 candidate margin 重算，并检查分母闭合。

In [2]:
levels = (pd.DataFrame(snapshot['evidence_rows'])
          .groupby('evidence_level', as_index=False)
          .agg(comparisons=('evidence_id', 'count'), games=('games', 'sum')))
print('证据层级与样本量')
print(levels.to_string(index=False))
print('\n自动发现状态')
print(f"fresh screen 比较数: {sum(r['evidence_level'] == 'fresh_screen' for r in snapshot['evidence_rows'])}")
print(f"fresh confirm 比较数: {sum(r['evidence_level'] == 'fresh_confirm' for r in snapshot['evidence_rows'])}")
print(f"online receipt 数: {len(snapshot['online_submissions'])}")
print(f"test outcomes read: {snapshot['scope']['test_outcomes_read']}")
print(f"new games started: {snapshot['scope']['new_games_started']}")
print(f"Kaggle API called: {snapshot['scope']['kaggle_api_called']}")

证据层级与样本量
evidence_level  comparisons  games
   dev_exposed            8    576
 fresh_confirm            2    400
  fresh_screen            2    144
oracle_exposed            2    272

自动发现状态
fresh screen 比较数: 2
fresh confirm 比较数: 2
online receipt 数: 1
test outcomes read: False
new games started: False
Kaggle API called: False


## Results: W/T/L、纯胜率与 score

In [3]:
metrics = pd.DataFrame(snapshot['evidence_rows'])[
    ['evidence_level', 'label', 'candidate', 'anchor', 'games', 'wins', 'ties', 'losses',
     'pure_win_rate', 'competition_score_rate', 'mean_margin', 'submission_eligible']
].copy()
metrics['W/T/L'] = metrics.apply(lambda r: f"{int(r.wins)}/{int(r.ties)}/{int(r.losses)}", axis=1)
metrics['pure_win_rate'] = metrics['pure_win_rate'].map(lambda x: f'{x:.2%}')
metrics['competition_score_rate'] = metrics['competition_score_rate'].map(lambda x: f'{x:.2%}')
metrics['mean_margin'] = metrics['mean_margin'].map(lambda x: f'{x:.3f}')
print(metrics.drop(columns=['wins', 'ties', 'losses']).to_string(index=False))

evidence_level                                                           label                              candidate             anchor  games pure_win_rate competition_score_rate mean_margin  submission_eligible     W/T/L
oracle_exposed                         Perfect-info oracle (V13C parent) vs A2  perfect_info_queue_oracle_v13c_parent v12a2_no_shop_gate    200        75.50%                 80.50%     822.565                False 151/20/29
oracle_exposed                           Perfect-info oracle (A2 parent) vs A2    perfect_info_queue_oracle_a2_parent v12a2_no_shop_gate     72        72.22%                 79.17%     672.306                False  52/10/10
   dev_exposed        S1 inventory-neutral WHEAT squeeze vs v12a2_no_shop_gate v14_s1_inventory_neutral_wheat_squeeze v12a2_no_shop_gate     72        62.50%                 70.83%       5.111                False  45/12/15
   dev_exposed        S1 inventory-neutral WHEAT squeeze vs v12_incumbent_r002 v14_s1_inventory_neutral_

### 按日 / 席位

优先展示 fresh confirm；若尚不存在，则展示 fresh screen；两者都不存在时只展示当前 Q2b 已暴露开发结果。这个回退规则不会改变证据标签。

In [4]:
available = pd.DataFrame(snapshot['evidence_rows'])
if (available['evidence_level'] == 'fresh_confirm').any():
    chosen_level = 'fresh_confirm'
    chosen_ids = set(available.loc[available.evidence_level == chosen_level, 'evidence_id'])
elif (available['evidence_level'] == 'fresh_screen').any():
    chosen_level = 'fresh_screen'
    chosen_ids = set(available.loc[available.evidence_level == chosen_level, 'evidence_id'])
else:
    chosen_level = 'dev_exposed (Q2b fallback)'
    chosen_ids = {'dev_q2b_vs_a2', 'dev_q2b_vs_r002'}

slices = pd.DataFrame([r for r in snapshot['slice_rows'] if r['evidence_id'] in chosen_ids]).copy()
slices['W/T/L'] = slices.apply(lambda r: f"{int(r.wins)}/{int(r.ties)}/{int(r.losses)}", axis=1)
slices['pure'] = slices['pure_win_rate'].map(lambda x: f'{x:.2%}')
slices['score'] = slices['competition_score_rate'].map(lambda x: f'{x:.2%}')
print(f'展示层级: {chosen_level}')
for dimension, label in (('date', '按日'), ('candidate_seat', '候选席位')):
    view = slices[slices.dimension == dimension][
        ['candidate', 'anchor', 'slice', 'games', 'W/T/L', 'pure', 'score', 'mean_margin']
    ]
    print(f'\n[{label}]')
    print(view.to_string(index=False) if not view.empty else '无可用切片')

展示层级: fresh_confirm

[按日]
                   candidate             anchor      slice  games    W/T/L   pure  score  mean_margin
v14_queue_stateful_no_mirror v12_incumbent_r002 2026-08-18     68  53/0/15 77.94% 77.94%   165.941176
v14_queue_stateful_no_mirror v12_incumbent_r002 2026-08-19     66  52/0/14 78.79% 78.79%   225.848485
v14_queue_stateful_no_mirror v12_incumbent_r002 2026-08-20     66  51/0/15 77.27% 77.27%   239.787879
v14_queue_stateful_no_mirror v12a2_no_shop_gate 2026-08-18     68 45/12/11 66.18% 75.00%   574.911765
v14_queue_stateful_no_mirror v12a2_no_shop_gate 2026-08-19     66  49/6/11 74.24% 78.79%   558.727273
v14_queue_stateful_no_mirror v12a2_no_shop_gate 2026-08-20     66  55/0/11 83.33% 83.33%  1016.090909

[候选席位]
                   candidate             anchor slice  games   W/T/L   pure  score  mean_margin
v14_queue_stateful_no_mirror v12_incumbent_r002     0    100 76/0/24 76.00% 76.00%       435.64
v14_queue_stateful_no_mirror v12_incumbent_r002     1    100

### Oracle gap：可达上限与已实现行为不能混为一谈

In [5]:
gap = snapshot['oracle_gap']
print(json.dumps(gap, ensure_ascii=False, indent=2))
print('\n解释：oracle 只回答“若知道完整未来，存在多大可达空间”；dev 结果回答“当前因果策略在已暴露面板做到多少”。')

{
  "games": 72,
  "wins": 52,
  "ties": 10,
  "losses": 10,
  "pure_win_rate": 0.7222222222222222,
  "competition_score_rate": 0.7916666666666666,
  "reordered_steps": 915,
  "shadow_faults": 0,
  "shadow_update_errors": 0,
  "all_shadow_trusted": true,
  "reward_vectors_exact_oracle": 64,
  "oracle_to_stateful": {
    "L->L": 10,
    "T->T": 10,
    "W->L": 5,
    "W->W": 47
  },
  "oracle_to_ablation": {
    "L->L": 10,
    "T->T": 10,
    "W->W": 52
  },
  "stateful_reordered_steps": 614,
  "ablation_reordered_steps": 915,
  "source_paths": [
    "kaggle_Kaggriculture/model/v14_first_principles_search/oracle_gap/public_equality_ablation.json",
    "kaggle_Kaggriculture/model/v14_first_principles_search/oracle_gap/task_alignment.json"
  ]
}

解释：oracle 只回答“若知道完整未来，存在多大可达空间”；dev 结果回答“当前因果策略在已暴露面板做到多少”。


### Shadow / calibration / package QA

In [6]:
print('Shadow exact-A2 QA')
print(json.dumps(snapshot['shadow_qa'], ensure_ascii=False, indent=2))
print('\nShadow calibration')
print(json.dumps(snapshot['shadow_calibration'], ensure_ascii=False, indent=2))
print('\nPackage QA')
print(pd.DataFrame(snapshot['packages']).to_string(index=False))
print('\n限制：这些 QA 证明 exact-A2 路径、实现或封装的一致性，不证明未知线上对手身份可识别。')

Shadow exact-A2 QA
{
  "candidate": "v14_queue_stateful_no_mirror",
  "games": 6,
  "compared_calls": 4314,
  "private_exact_calls": 4314,
  "private_exact_rate": 1.0,
  "action_exact_calls": 4314,
  "action_exact_rate": 1.0,
  "day_boundary_calls": 174,
  "day_boundary_private_exact_calls": 174,
  "day_boundary_action_exact_calls": 174,
  "reordered_steps": 76,
  "shadow_faults": 0,
  "shadow_update_errors": 0,
  "go": true,
  "scope": "exact A2 only; not unknown-opponent identification",
  "source_path": "kaggle_Kaggriculture/model/v14_first_principles_search/shadow_qa/exact_a2_shadow_q2b_qa.json"
}

Shadow calibration
{
  "games": 272,
  "opportunities": 13581,
  "method": "flow_corrected",
  "queue_exact_rate": 1.0,
  "shed_exact_rate": 1.0,
  "exec_cap_exact_rate": 1.0,
  "gate": "flow_history",
  "gate_coverage": 0.9265149841690598,
  "gate_queue_precision": 1.0,
  "gate_queue_wilson95_lower": 0.999694803590459,
  "epistemic_status": "offline action/private-label calibration on a

### Community findings（机制输入，不是本地胜率证据）

In [7]:
community = pd.DataFrame(snapshot['community_findings'])
print(community.to_string(index=False))

                        topic                                 finding                                evidence_status
            737128 MELON tree                观测式未来价格预测不等于动作反事实；不直接移植。 community self-report + local mechanism review
    737027 opponent inventory 可识别商品应做区间/残差跟踪；地板、DROP、overflow 时保持不确定。                          community self-report
736439 non-transitive experts  支持条件 payoff / best response，不支持静态平均混合。                          community self-report
      734412 market semantics               共享市场、需求钟与 slot 顺序是直接交互通道。       community + official-engine verification


## Hash closure 与来源清单

In [8]:
closure = pd.DataFrame(snapshot['hash_closure'])
declared = closure[closure['expected_sha256'].notna()].copy()
bad = declared[declared['status'] != 'pass']
print(f"声明型 hash 检查: {len(declared)}；失败: {len(bad)}；闭包通过: {snapshot['submission_decision']['declared_hash_closure_pass']}")
if not declared.empty:
    print(declared[['role', 'path', 'expected_sha256', 'actual_sha256', 'status']].to_string(index=False))
print(f"\n实际读取来源: {len(snapshot['source_inventory'])} 个文件；每个文件均记录 SHA-256。")
inventory = pd.DataFrame(snapshot['source_inventory'])
print(inventory[['path', 'sha256', 'size_bytes']].to_string(index=False))

声明型 hash 检查: 36；失败: 0；闭包通过: True
                                                                 role                                                                                                                               path                                                  expected_sha256                                                    actual_sha256 status
                                     oracle_v13c_vs_a2 declared games                                                   kaggle_Kaggriculture/model/v14_first_principles_search/oracle/strict_games.jsonl ccb929b9953d77f683e045a1e536668542196ef7b0f4912b67a73f1e67410317 ccb929b9953d77f683e045a1e536668542196ef7b0f4912b67a73f1e67410317   pass
                                oracle_a2_parent_vs_a2 declared games                                         kaggle_Kaggriculture/model/v14_first_principles_search/oracle/a2_parent_oracle_games.jsonl a8d0482aacdc2272cf1e68138412aad1731745d056a1657481c3f5baafa46554 a8d0482aacdc2272cf1e68138

## Takeaways

In [9]:
decision = snapshot['submission_decision']
print(f"当前状态: {snapshot['overall_status']}")
print(f"Q2b package ready: {decision['package_ready_q2b']}")
print(f"declared hash closure: {decision['declared_hash_closure_pass']}")
print(f"fresh confirm pass candidates: {decision['fresh_confirm_candidates_passing'] or '无'}")
print(f"online records: {decision['online_record_count']}")
print(f"unknown-opponent identity proven: {decision['unknown_opponent_identity_proven']}")
if not decision['fresh_confirm_candidates_passing']:
    print('结论：当前只能确认开发机制和工程闭包；尚不能宣称 fresh 65% 门通过，更不能宣称线上有效。')
elif not snapshot['online_submissions']:
    print('结论：已有 fresh confirm 本地提交依据，但没有 online receipt，不能宣称线上有效。')
else:
    print('结论：已有线上 receipt；仍需按 receipt 的状态与 rating 单独解释，不能用单次初始值外推稳定排名。')

当前状态: online evidence present; inspect Kaggle status separately
Q2b package ready: True
declared hash closure: True
fresh confirm pass candidates: ['v14_queue_stateful_no_mirror']
online records: 1
unknown-opponent identity proven: False
结论：已有线上 receipt；仍需按 receipt 的状态与 rating 单独解释，不能用单次初始值外推稳定排名。
